# NFL Big Data Bowl 2027 — Phase 2B | WR Combine-Only Movement Feasibility

**Research checkpoint:** Verify if DAGGER / SLANT / GO route trajectories have distinguishable, stable movement transitions **without looking at NFL regular-season outcomes**.

This exploratory notebook deliberately does **not** perform predictive modeling or Combine-to-NFL correlation analysis. Numerical thresholds are preliminary, not learned or optimized against outcomes.

**Data:** official `combine_tracking.csv` (or uploaded renamed copy).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.stats import spearmanr

# Edit this path for Kaggle or local environment.
CANDIDATES = [
    Path('/mnt/data/combine_tracking(1).csv'),
    Path('combine_tracking.csv'),
    Path('combine_tracking(1).csv'),
]
try:
    DATA_PATH = next(p for p in CANDIDATES if p.exists())
except StopIteration:
    raise FileNotFoundError('Set DATA_PATH to the competition combine_tracking.csv')
print('Reading:', DATA_PATH)
raw = pd.read_csv(DATA_PATH)
DRILLS = ['SLANT_ROUTE_LEFT','DAGGER_ROUTE_LEFT','GO_ROUTE_RIGHT']
wr_all = raw.loc[(raw['entity_type']=='PLAYER') & (raw['drill_type']=='SKILL_DRILLS_WR')].copy()
wr_all['timestamp'] = pd.to_datetime(wr_all['time'])
wr_all = wr_all.sort_values(['event_id', 'timestamp'])
wr = wr_all.loc[wr_all['drill_name'].isin(DRILLS)].copy()
print(f'WR: {len(wr_all):,} frames; {wr_all.event_id.nunique():,} attempts; {wr_all.nfl_id.nunique()} prospects')
print('Selected drills:', wr.groupby('drill_name').agg(attempts=('event_id','nunique'), players=('nfl_id','nunique'),frames=('event_id','size')))

## A. Input quality & drill coverage

`event_id` identifies continuous attempt; `nfl_id` identifies player. Frame spacing should be 0.1 seconds. Distinguish sampled frames from independent athletes.

In [ ]:
quality=[]
for event_id, g in wr_all.groupby('event_id', sort=False):
    dt = g.timestamp.diff().dt.total_seconds().dropna().to_numpy()
    quality.append({
        'event_id':event_id,'nfl_id':int(g.nfl_id.iloc[0]),
        'draft_year':int(g.draft_year.iloc[0]),
        'drill_name':g.drill_name.iloc[0],
        'frames':len(g), 'duration_s': (g.timestamp.iloc[-1]-g.timestamp.iloc[0]).total_seconds(),
        'bad_intervals':int(np.sum(np.abs(dt-0.1)>0.011)),
        'duplicate_times':int(g.timestamp.duplicated().sum()),
        'one_player':g.nfl_id.nunique()==1,
    })
qc = pd.DataFrame(quality)
print('Total anomalous intervals:', qc.bad_intervals.sum())
print('Total duplicate timestamps:', qc.duplicate_times.sum())
print('Attempts with multiple nfl_ids:', (~qc.one_player).sum())
print(qc[qc.drill_name.isin(DRILLS)].groupby(['drill_name','draft_year']).agg(attempts=('event_id','count'), median_duration=('duration_s','median')))

## B. Provisional transition detector

1. Unwrap circular `dir` angles and smooth using a 7-frame Savitzky–Golay filter (alternatives 5 and 9).
2. Detect maximum **absolute heading change over 0.6 seconds** in the middle 25–78% of the attempt, while speed ≥ 4 yd/s.
3. Calculate a provisional **speed-loss proxy**: `1 - minimum_speed_near_transition / maximum_speed_in_pre-transition_window`.
4. A speed-loss proxy below zero indicates speed increasing; it is **not** a negative physical braking force.
5. GO is a drill-structure negative control, not a causal control group. Do not transfer route-specific metrics without checking comparability.

In [ ]:
def transition_metrics(g, smooth_window=7, half_width_frames=3):
    t = (g.timestamp - g.timestamp.iloc[0]).dt.total_seconds().to_numpy()
    s = g.s.to_numpy(); heading = np.unwrap(np.deg2rad(g.dir.to_numpy()))
    n = len(g); w = smooth_window; h = half_width_frames
    if n < max(w, 2*h+2):
        return None
    hs = savgol_filter(heading, w, 2, mode='interp')
    ss = savgol_filter(s, w, 2, mode='interp')
    change = np.full(n,np.nan)
    change[h:n-h] = np.abs(np.rad2deg(hs[2*h:] - hs[:n-2*h]))
    valid = ((np.arange(n)>=int(.25*(n-1))) &
             (np.arange(n)<=int(.78*(n-1))) &
             (ss>=4.) & np.isfinite(change))
    if not np.any(valid):
        return None
    j = int(np.nanargmax(np.where(valid,change,np.nan)))
    pre = ss[max(0,j-10):max(1,j-2)]
    around = ss[max(0,j-2):min(n,j+3)]
    post = ss[min(n,j+3):min(n,j+13)]
    if not len(pre) or not len(around) or not len(post):
        return None
    peak_pre=float(np.max(pre)); cut_min=float(np.min(around)); peak_post=float(np.max(post))
    ret = cut_min / peak_pre if peak_pre>0 else np.nan
    return {'cut_time_s':t[j],'cut_fraction': t[j]/t[-1],
            'turn_deg_0p6s':float(change[j]),'pre_speed':peak_pre,
            'cut_min_speed':cut_min,'post_speed':peak_post,
            'speed_loss_proxy':1-ret,'post_pre_speed_ratio':peak_post/peak_pre,
            'passes_exploratory_rule':bool(change[j]>=25 and .05<ret<1.25)}

records=[]
for event_id,g in wr.groupby('event_id'):
    for win in [5,7,9]:
        metrics=transition_metrics(g, smooth_window=win)
        if metrics is not None:
            records.append({'event_id':event_id,'nfl_id':int(g.nfl_id.iloc[0]),
                            'draft_year':int(g.draft_year.iloc[0]),
                            'drill_name':g.drill_name.iloc[0],
                            'smooth_window':win,'duration_s':(g.timestamp.iloc[-1]-g.timestamp.iloc[0]).total_seconds(), **metrics})
diagnostic=pd.DataFrame(records)
main=diagnostic[diagnostic.smooth_window==7].copy()
print(main.groupby('drill_name').agg(attempts=('event_id','count'),
       median_turn=('turn_deg_0p6s','median'),
       median_loss=('speed_loss_proxy','median'),passes=('passes_exploratory_rule','sum')).round(3))

## C. Movement trajectories and distributions

These are directly based on player sensor frames, not schematic football routes. The marked transition is an algorithm output, not manually annotated ground truth.

In [ ]:
player_id = int(wr[wr.draft_year.eq(2024)].nfl_id.iloc[0])
# Select an athlete with all three drills to ensure comparability.
counts = wr.groupby('nfl_id').drill_name.nunique()
players3 = sorted(counts[counts.eq(3)].index.tolist())
player_id = next(p for p in players3 if int(wr.loc[wr.nfl_id.eq(p),'draft_year'].iloc[0])==2024)
fig,ax=plt.subplots(3,2,figsize=(10,11))
for i,name in enumerate(DRILLS):
    g=wr[(wr.nfl_id==player_id)&wr.drill_name.eq(name)]
    g=g[g.event_id.eq(g.event_id.iloc[0])]
    ts=(g.timestamp-g.timestamp.iloc[0]).dt.total_seconds().to_numpy()
    x=g.x.to_numpy();y=g.y.to_numpy()
    ax[i,0].plot(x-x[0],y-y[0]);ax[i,0].scatter(0,0,s=24)
    ax[i,0].set_title(name+' — relative trajectory');ax[i,0].set_xlabel('Δx yards');ax[i,0].set_ylabel('Δy yards')
    ax[i,1].plot(ts,g.s.to_numpy());ax[i,1].set_title('Speed vs time');ax[i,1].set_xlabel('Seconds');ax[i,1].set_ylabel('Yards per second')
fig.suptitle(f'Representative combined 2024 athlete (nfl_id={player_id})')
fig.tight_layout();plt.show()

fig,axs=plt.subplots(1,2,figsize=(10,4))
for ax,col in zip(axs,['turn_deg_0p6s','speed_loss_proxy']):
    series=[main.loc[main.drill_name.eq(name),col].dropna() for name in DRILLS]
    ax.boxplot(series,tick_labels=['SLANT','DAGGER','GO'],showfliers=False)
    ax.set_title(col.replace('_',' '));ax.grid(axis='y',alpha=.15)
plt.tight_layout();plt.show()

## D. Sensitivity and repeatability limitations

Smooth-window sensitivity is **not** test–retest reliability. There are very few repeated attempts per player. Different draft-year recording windows may affect metric comparability.

In [ ]:
for name in DRILLS:
    z=diagnostic[diagnostic.drill_name.eq(name)]
    base=z[z.smooth_window.eq(7)].set_index('event_id')
    print(); print(name)
    for win in [5,9]:
        alt=z[z.smooth_window.eq(win)].set_index('event_id')
        joined=base[['cut_time_s','turn_deg_0p6s','speed_loss_proxy']].join(
            alt[['cut_time_s','turn_deg_0p6s','speed_loss_proxy']],lsuffix='_7',rsuffix='_alt')
        dtime=np.abs(joined.cut_time_s_7 - joined.cut_time_s_alt)
        print('window',win, 'pct cut within 0.3s:',round(float((dtime<=.31).mean()),3),
              'rank rho speed_loss:',round(float(spearmanr(joined.speed_loss_proxy_7,joined.speed_loss_proxy_alt,nan_policy='omit').statistic),3))
    repeat=base.groupby('nfl_id').size()
    print('players with >=2 attempts:',int((repeat>=2).sum()))
    print('median durations by year:',base.groupby('draft_year').duration_s.median().to_dict())

## E. Checkpoint decision

**GO WITH REVISION (provisional):** prioritize DAGGER transition biomechanics. SLANT should not be interpreted as braking; GO serves as movement-structure comparison. Before locking the final NFL research question, perform independent/manual event-window review, validate trajectory-based heading extraction, and explicitly justify the NFL outcome and benchmark design.

**Not established:** NFL linkage; cross-player causal effect; test–retest reliability; a winning/novel scouting metric.

**Next:** Phase 3 research protocol, including outcome definition and cohort selection (after the Combine-only metric specification is frozen).